# ATHLT Basketball Shot Detector

Trains YOLOv11n on a basketball detection dataset and exports to CoreML for on-device iOS inference.

**Before running:** Set runtime to T4 GPU — `Runtime → Change runtime type → T4 GPU`

**Expected time:** 45–90 min on free T4

## 1 — Check GPU

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print('GPU:', r.stdout.strip())
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU and re-run.')
    sys.exit('No GPU available')

## 2 — Install packages

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics, torch
print('ultralytics:', ultralytics.__version__)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 3 — Download dataset

Primary: `basketball-6vyfz/basketball-detection-srfkd` — the dataset used by SwishAI.
Classes: Ball, Ball_in_Basket, Player, Basket, Player_Shooting

Fallback datasets are tried automatically if the primary fails.

In [ ]:
from roboflow import Roboflow
import os, yaml

API_KEY = 'zOij8uPk918MwiTSFuVN'
rf = Roboflow(api_key=API_KEY)

# Verified datasets (workspace, project, version, description)
# Ordered by preference: most classes first, most images first
CANDIDATES = [
    # Primary — 5-class SwishAI dataset: Ball, Ball_in_Basket, Player, Basket, Player_Shooting
    ('basketball-6vyfz',         'basketball-detection-srfkd',              1, '5-class SwishAI dataset'),
    ('basketball-6vyfz',         'basketball-detection-srfkd',              2, '5-class SwishAI dataset v2'),
    ('basketball-6vyfz',         'basketball-detection-srfkd',              3, '5-class SwishAI dataset v3'),
    # Fallback A — Official Roboflow: ball, rim, player + action classes (~654 images)
    ('roboflow-jvuqo',           'basketball-player-detection-3-ycjdo',     6, 'Official Roboflow ball+rim'),
    ('roboflow-jvuqo',           'basketball-player-detection-3-ycjdo',     5, 'Official Roboflow ball+rim v5'),
    # Fallback B — simple 3-class: person, ball, basketball hoop (~657 images)
    ('basketball-detection-b977c','basketball-detection-sskux',             7, '3-class ball+hoop'),
    ('basketball-detection-b977c','basketball-detection-sskux',             6, '3-class ball+hoop v6'),
    # Fallback C — basketball players with hoop class
    ('roboflow-universe-projects','basketball-players-fy4c2',              22, 'Ball+Hoop+Player'),
]

dataset = None
chosen_desc = ''

for (workspace, project_slug, version_num, desc) in CANDIDATES:
    try:
        print(f'Trying {workspace}/{project_slug} v{version_num} ({desc})...')
        project = rf.workspace(workspace).project(project_slug)
        version_obj = project.version(version_num)
        # Use yolov8 format — same label format as yolov11, fully compatible
        dataset = version_obj.download('yolov8')
        chosen_desc = desc
        print(f'  Downloaded successfully to: {dataset.location}')
        break
    except Exception as e:
        print(f'  Failed: {type(e).__name__}: {str(e)[:120]}')
        continue

if dataset is None:
    raise RuntimeError(
        'All dataset candidates failed.\n'
        'Check that your API key is valid and your Roboflow account has access.\n'
        'API key used: ' + API_KEY[:8] + '...'
    )

print(f'\nUsing dataset: {chosen_desc}')
print(f'Location: {dataset.location}')

In [ ]:
# Read and display the data.yaml to confirm classes
data_yaml = os.path.join(dataset.location, 'data.yaml')
assert os.path.exists(data_yaml), f'data.yaml not found at {data_yaml}'

with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

print('Class names:', cfg.get('names', []))
print('Num classes:', cfg.get('nc', '?'))

for split in ('train', 'valid', 'test'):
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.isdir(img_dir):
        n = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f'{split}: {n} images')

## 4 — Train YOLOv11n

In [ ]:
from ultralytics import YOLO

# YOLOv11n — fastest (~17ms/frame on iPhone 13), 6MB, good enough for ball+rim detection
model = YOLO('yolo11n.pt')
print('Base model loaded.')

In [ ]:
results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,           # lower to 8 if CUDA OOM
    device=0,
    project='runs/detect',
    name='athlt_v1',
    patience=15,        # early stop if no improvement for 15 epochs
    save=True,
    save_period=10,
    plots=True,
    # augmentation for basketball footage
    hsv_s=0.7,
    shear=2.0,
    mixup=0.1,
    flipud=0.0,
    fliplr=0.5,
)

BEST_PT = str(results.save_dir) + '/weights/best.pt'
print('Training done. Best weights:', BEST_PT)

## 5 — Validate

In [ ]:
best = YOLO(BEST_PT)
val = best.val(data=data_yaml, device=0)

print(f'mAP@50:    {val.box.map50:.4f}')
print(f'mAP@50-95: {val.box.map:.4f}')
print(f'Precision: {val.box.mp:.4f}')
print(f'Recall:    {val.box.mr:.4f}')
print('Per-class AP@50:')
for i, name in enumerate(val.names.values()):
    ap = val.box.ap50[i] if i < len(val.box.ap50) else float('nan')
    print(f'  {name}: {ap:.4f}')

## 6 — Sample detections (inline)

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image as PILImage
from IPython.display import Image as IPYImage, display

COLORS = {
    'ball':             '#FFD700',
    'Ball':             '#FFD700',
    'ball_in_basket':   '#00FF88',
    'Ball_in_Basket':   '#00FF88',
    'basket':           '#00BFFF',
    'Basket':           '#00BFFF',
    'rim':              '#00BFFF',
    'hoop':             '#00BFFF',
    'basketball hoop':  '#00BFFF',
    'player':           '#FF8C00',
    'Player':           '#FF8C00',
    'player_shooting':  '#FF1493',
    'Player_Shooting':  '#FF1493',
    'person':           '#FF8C00',
}

val_imgs = (glob.glob(os.path.join(dataset.location, 'valid', 'images', '*.jpg')) +
            glob.glob(os.path.join(dataset.location, 'valid', 'images', '*.png')))[:4]

if val_imgs:
    fig, axes = plt.subplots(1, len(val_imgs), figsize=(5 * len(val_imgs), 5))
    if len(val_imgs) == 1:
        axes = [axes]
    for ax, p in zip(axes, val_imgs):
        img = PILImage.open(p)
        preds = best.predict(p, conf=0.30, verbose=False)
        ax.imshow(np.array(img)); ax.axis('off')
        for pred in preds:
            for box in pred.boxes:
                x1,y1,x2,y2 = box.xyxy[0].tolist()
                name  = pred.names[int(box.cls[0])]
                conf  = float(box.conf[0])
                color = COLORS.get(name, '#FFFFFF')
                ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                    linewidth=2,edgecolor=color,facecolor='none'))
                ax.text(x1, y1-3, f'{name} {conf:.2f}', color=color, fontsize=8,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2',facecolor='black',alpha=0.6))
    plt.tight_layout()
    plt.savefig('/content/sample_detections.png', dpi=100)
    plt.show()
else:
    print('No validation images found.')

## 7 — Export to CoreML

In [ ]:
!pip install -q coremltools
import coremltools as ct
print('coremltools:', ct.__version__)

In [ ]:
# Export to CoreML. nms=True bakes NMS into the model (simpler device-side code).
# Falls back to nms=False if the NMS export fails (happens on some coremltools versions).
export_path = None

for nms_flag in (True, False):
    try:
        print(f'Exporting CoreML (nms={nms_flag})...')
        export_path = best.export(
            format='coreml',
            nms=nms_flag,
            imgsz=640,
            half=False,   # FP32 — better compatibility across iPhone models
        )
        print(f'Export succeeded: {export_path}')
        break
    except Exception as e:
        print(f'  nms={nms_flag} failed: {e}')
        continue

if export_path is None or not os.path.exists(str(export_path)):
    raise RuntimeError('CoreML export failed both with and without NMS. Check coremltools version.')

# Compute total size
total = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk(export_path)
    for f in files
)
print(f'Model size: {total/1024/1024:.1f} MB')

## 8 — Rename to best.mlpackage and download

In [ ]:
import shutil

FINAL = '/content/best.mlpackage'
ZIP   = '/content/best_mlpackage.zip'

# Copy to standardised name
if os.path.abspath(str(export_path)) != os.path.abspath(FINAL):
    if os.path.exists(FINAL):
        shutil.rmtree(FINAL)
    shutil.copytree(str(export_path), FINAL)

assert os.path.exists(FINAL), 'best.mlpackage not found after copy'
print('Verified best.mlpackage exists.')

# Zip it (directory, not single file)
if os.path.exists(ZIP):
    os.remove(ZIP)
shutil.make_archive('/content/best_mlpackage', 'zip', '/content', 'best.mlpackage')

size_mb = os.path.getsize(ZIP) / 1024 / 1024
print(f'Zipped: {ZIP} ({size_mb:.1f} MB)')

In [ ]:
from google.colab import files
print('Triggering download of best_mlpackage.zip ...')
files.download(ZIP)

## 9 — What to do with the downloaded file

1. **Unzip** `best_mlpackage.zip` — you get a folder called `best.mlpackage`
2. **Move** it to `expo/cv/best.mlpackage` in your ATHLT project
3. **Activate the config plugin** in `expo/app.json` — uncomment the `withCoreMLModel` entry
4. **Run** `eas build --platform ios --profile development`

Do NOT commit `best.mlpackage` to git — add it to `.gitignore`.

---

### If Colab disconnects mid-training

Re-run cells 1–3, then use this to resume:

In [ ]:
# RESUME ONLY — run this cell instead of the train cell if training was interrupted
# from ultralytics import YOLO
# BEST_PT = 'runs/detect/athlt_v1/weights/last.pt'
# model = YOLO(BEST_PT)
# results = model.train(resume=True)